# FreightQuote AI — Milestone 2
Full-Stack AI/ML Integration & Advanced Security Engine

**Repository:** https://github.com/Kavyashree1203/Infosys_FreightQuote_AI/tree/main/Milestone2

This notebook wires together the Security Gateway (auth.py, db.py), the 3 ML agents
(train_ml_freight.py), the LLM Copilot (llm_engine_freight.py), and the Admin
Dashboard (admin_dash.py) behind a single Streamlit app (app.py), tunneled with ngrok.


## Step 0 — Confirm GPU (Section 3.1)

In [ ]:
!nvidia-smi

## Step 1 — Get the Milestone2 code
Clones the Infosys_FreightQuote_AI repo directly from GitHub and moves into the
Milestone2 folder. If you've already cloned it once in this Drive, this just cds in.

In [ ]:
import os

REPO_URL = "https://github.com/Kavyashree1203/Infosys_FreightQuote_AI.git"
REPO_DIR = "/content/Infosys_FreightQuote_AI"
PROJECT_DIR = f"{REPO_DIR}/Milestone2"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    print("Repo already cloned, pulling latest changes...")
    !cd $REPO_DIR && git pull

%cd $PROJECT_DIR
!ls

## Step 2 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Step 3 — Load Colab Secrets (Section 3.3)
Click the 🔑 key icon in the left sidebar and add each secret listed in the instructions
(JWT_SECRET_KEY, ADMIN_EMAIL_ID, ADMIN_PASSWORD, NGROK_AUTHTOKEN, HF_TOKEN, EMAIL_ID,
EMAIL_PASSWORD, KAGGLE_USERNAME, KAGGLE_KEY) before running this cell.

In [ ]:
from google.colab import userdata
import os

JWT_SECRET_KEY   = userdata.get('JWT_SECRET_KEY')
ADMIN_EMAIL_ID   = userdata.get('ADMIN_EMAIL_ID') or "infosys@ai"
ADMIN_PASSWORD   = userdata.get('ADMIN_PASSWORD') or "admin@123"
NGROK_AUTHTOKEN  = userdata.get('NGROK_AUTHTOKEN')
HF_TOKEN         = userdata.get('HF_TOKEN')

try:
    EMAIL_ID = userdata.get('EMAIL_ID')
except Exception:
    EMAIL_ID = None
try:
    EMAIL_PASSWORD = userdata.get('EMAIL_PASSWORD')
except Exception:
    EMAIL_PASSWORD = None

# Kaggle (optional — Section 3.2)
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    KAGGLE_CONFIGURED = True
except Exception:
    KAGGLE_CONFIGURED = False

# HuggingFace login for Qwen2.5-3B
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

print("Secrets loaded. Kaggle configured:", KAGGLE_CONFIGURED)

## Step 4 — Initialize DB, seed Admin account, configure auth.py

In [ ]:
import db, auth

db.init_db()
auth.configure(jwt_secret=JWT_SECRET_KEY, email_id=EMAIL_ID, email_password=EMAIL_PASSWORD)
db.seed_admin(admin_email=ADMIN_EMAIL_ID, admin_password_hash=auth.hash_password(ADMIN_PASSWORD))

print(f"Admin account ready -> email: {ADMIN_EMAIL_ID}")

## Step 5 — Train all 3 ML agents (Section 7)
Pulls the Kaggle datasets in Section 7.1 if `KAGGLE_CONFIGURED`, otherwise trains on the
seeded synthetic data fallback. Each agent compares 5+ algorithms and saves a champion
model + logs every metric to the `ml_models` table.

In [ ]:
import train_ml_freight as trainer

# Optional: attempt real Kaggle datasets per Section 7.1 (falls back to synthetic if unavailable)
agent1_df = None
agent2_df = None
agent3_df = None

if KAGGLE_CONFIGURED:
    import pandas as pd

    p1 = trainer.try_kaggle_download(
        "apoorvwatsky/supply-chain-shipmentpricing-data", "SCMS_Delivery_History_Dataset.csv")
    if p1:
        print("Loaded Agent 1 Kaggle dataset:", p1)
        # NOTE: adapt column selection/renaming here to match the CSV schema
        # agent1_df = pd.read_csv(p1)

    p2 = trainer.try_kaggle_download(
        "harshsingh2209/supply-chain-analysis", "supply_chain_data.csv")
    if p2:
        print("Loaded Agent 2 Kaggle dataset:", p2)
        # agent2_df = pd.read_csv(p2)

    p3 = trainer.try_kaggle_download(
        "davidcariboo/freight-carrier-performance", "carrier_perf.csv")
    if p3:
        print("Loaded Agent 3 Kaggle dataset:", p3)
        # agent3_df = pd.read_csv(p3)

results = trainer.train_all_agents()
print(results)

## Step 6 — Load the Qwen2.5-3B (4-bit) LLM Copilot (Section 8)

In [ ]:
import llm_engine_freight as llm

llm_ready = llm.load_llm()
print("LLM active:", llm_ready)

## Step 7 — Quick sanity test of the Copilot (Section 8)
Try the exact prompt from the instructions before moving on.

In [ ]:
test_context = {
    "route": "Shanghai-Rotterdam", "cost": 4200,
    "delay_risk_pct": 68, "congestion": "high", "canal_queue": True,
    "carrier": "Maersk", "punctuality_rate": 94, "compliance_risk": "moderate",
}
print(llm.ask_copilot("Explain in 2 sentences why port congestion increases freight risk.", test_context))

## Step 8 — Launch Streamlit + ngrok tunnel (Section 10.1)

In [ ]:
from pyngrok import ngrok
import subprocess, time

ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Kill any previous tunnels/streamlit instances
ngrok.kill()
!pkill -f streamlit || true
time.sleep(2)

process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(8)

public_url = ngrok.connect(8501)
print("🌐 FreightQuote AI is live at:", public_url)

## Step 9 — Verify checklist (Section 10.1)
- [ ] Login works with ADMIN_EMAIL_ID / ADMIN_PASSWORD
- [ ] Home page / KPI overview loads
- [ ] AI Copilot returns a real response (or documented fallback)
- [ ] Agent 1 Pricing Calculator returns a predicted cost
- [ ] Admin Panel → ML Model Card shows R²/ROC-AUC for all 3 agents
- [ ] Progressive lockout, OTP cooldown, password strength badges behave as specified


## Step 10 — Finalize (Section 10.2)
Before uploading:
1. Restart runtime and re-run top-to-bottom.
2. Edit → Clear all outputs.
3. Search this notebook for any hard-coded secrets and remove them — only
   `userdata.get(...)` lookups should remain.
4. Download as .ipynb and place inside the Milestone2 folder.


In [ ]:
# Stop the tunnel/server when you're done capturing screenshots
# ngrok.kill()
# process.terminate()